# Importar Bibliotecas

In [11]:
import logging
import os
import pandas as pd
import re
import requests
import time
import urllib3
from bs4 import BeautifulSoup
from datetime import datetime
from pathlib import Path
from tqdm import tqdm # Para barra de progresso visual
from typing import List, Dict, Any
from urllib3.exceptions import InsecureRequestWarning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configurações e Constantes

In [12]:
BASE_URL = 'https://investidor10.com.br/fiis/'

USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'

REQUEST_TIMEOUT = 15
SLEEP_TIME_SECONDS = 1.5
MAX_RETRIES = 3

# Caminhos de Saída
OUTPUT_DIR = Path(r'C:\Users\RODRIGO\OneDrive\Documentos\Investimentos')
OUTPUT_FILENAME = 'Dados_FIIs_Indicadores.xlsx'
OUTPUT_DIVIDENDS_FILENAME = 'Dados_FIIs_Rendimentos.xlsx'

# Lista de FIIs

In [13]:
FIIS_LIST = [
    'AFHI11','ALZR11',
    'BIME11','BRCO11','BTCI11','BTLG11',
    'CPTS11','CYCR11',
    'FATN11','FIIP11','FYTO11',
    'GGRC11',
    'HGBS11','HGLG11','HGRU11','HSLG11','HSML11','HTMX11',
    'IFRA11',
    'JSRE11',
    'KCRE11','KNCA11','KNCR11','KNHF11','KNRI11','KNSC11','KNUQ11',
    'MCCI11','MXRF11',
    'PCIP11','PMLL11','PORD11','PVBI11',
    'RBRL11','RBRR11','RBRY11','RBVA11','RCRB11','RECR11','RZAT11',
    'TEPP11','TGAR11',
    'VCJR11','VGIP11','VISC11','VILG11','VRTA11',
    'XPCI11','XPLG11','XPML11',
    'ZAVI11'
]

# Funções de Auxílio

In [14]:
def fetch_page(ticker):
    # Desabilita avisos de SSL inseguro
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)    
    url = f"{BASE_URL}{ticker}/"
    headers = {'User-Agent': USER_AGENT}
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT, verify=False)
            response.raise_for_status()
            print(f'Código de Status: {response.status_code}')
            print('\nTrecho do HTML (primeiros 1000 caracteres):')
            print(response.text[:1000])
            return response
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(SLEEP_TIME_SECONDS * (attempt + 1))
            else:
                return None

def clean_numeric(value_str):
    """Converte valores como 'R$ 1.234,56' ou '10,5%' em float"""
    if not value_str or value_str == "N/A" or value_str == "-":
        return None
    try:
        clean_val = value_str.replace('R$', '').replace('%', '').replace('.', '').replace(',', '.').strip()
        return float(clean_val)
    except:
        return value_str

def scrape_indicators(soup, ticker):
    table_indicators_div = soup.find('div', id='table-indicators')
    fii_data = {'Ticker': ticker}
    
    if table_indicators_div:
        cells = table_indicators_div.find_all('div', class_='cell')
        for cell in cells:
            label_elem = cell.find('span', class_='name')
            value_elem = cell.find('div', class_='value')
            if label_elem and value_elem:
                label = re.sub(r'\s+', ' ', label_elem.text.strip()).upper()
                value = value_elem.text.strip()
                # Aplicamos limpeza apenas se for um campo que sabemos ser numérico
                if any(x in label for x in ['VALOR', 'COTA', 'RENDIMENTO', 'VACÂNCIA', 'NÚMERO']):
                    fii_data[label] = clean_numeric(value)
                else:
                    fii_data[label] = value
    return fii_data

def scrape_dividends(soup, ticker):
    dividends_list = []
    table = soup.find('table', id='table-dividends-history')
    
    if not table:
        return dividends_list

    current_year = datetime.now().year
    years_to_scrape = range(current_year - 5, current_year + 1)
    
    rows = table.find_all('tr')[1:] # Pula cabeçalho
    for row in rows:
        cols = row.find_all('td')
        if len(cols) >= 4:
            # Estrutura: Tipo, Data COM, Pagamento, Valor
            date_str = cols[2].get_text(strip=True)
            value_str = cols[3].get_text(strip=True)
            
            try:
                if date_str and date_str != "-":
                    pay_date = datetime.strptime(date_str, '%d/%m/%Y')
                    if pay_date.year in years_to_scrape:
                        dividends_list.append({
                            'Ticker': ticker,
                            'Data Pagamento': pay_date.strftime('%Y-%m-%d'),
                            'Rendimento': clean_numeric(value_str)
                        })
            except:
                continue
    return dividends_list

# Execução Principal

In [15]:
def main():
    all_indicators = []
    all_dividends = []
    
    print(f"Iniciando coleta de {len(FIIS_LIST)} FIIs...")
    
    for ticker in tqdm(FIIS_LIST, desc="Progresso"):
        response = fetch_page(ticker)
        
        if response:
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # 1. Indicadores
            data = scrape_indicators(soup, ticker)
            all_indicators.append(data)
            
            # 2. Dividendos
            divs = scrape_dividends(soup, ticker)
            all_dividends.extend(divs)
        else:
            all_indicators.append({'Ticker': ticker, 'Status': 'Falha ao acessar'})
            
        time.sleep(SLEEP_TIME_SECONDS)

    # --- Exportação ---
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # Salvar Indicadores
    if all_indicators:
        df_ind = pd.DataFrame(all_indicators)
        # Reordenar colunas principais primeiro (se existirem)
        cols_priority = ['Ticker', 'SEGMENTO', 'TIPO DE FUNDO', 'VACÂNCIA', 'ÚLTIMO RENDIMENTO']
        available = [c for c in cols_priority if c in df_ind.columns]
        remaining = [c for c in df_ind.columns if c not in available]
        df_ind = df_ind[available + remaining]
        
        df_ind.to_excel(OUTPUT_DIR / OUTPUT_FILENAME, index=False)
        print(f"\nIndicadores salvos em: {OUTPUT_FILENAME}")

    # Salvar Dividendos
    if all_dividends:
        df_div = pd.DataFrame(all_dividends)
        df_div.to_excel(OUTPUT_DIR / OUTPUT_DIVIDENDS_FILENAME, index=False)
        print(f"Rendimentos salvos em: {OUTPUT_DIVIDENDS_FILENAME}")

if __name__ == "__main__":
    main()

Iniciando coleta de 51 FIIs...


Progresso:   0%|          | 0/51 [00:00<?, ?it/s]

Código de Status: 200

Trecho do HTML (primeiros 1000 caracteres):
<!DOCTYPE html>
<html lang="pt-BR">

<head>
    <meta http-equiv="x-ua-compatible" content="ie=edge" />
    <meta charset="UTF-8" />
    <meta name="language" content="pt-BR" />
    <meta name="robots" content="index,follow">

    <title>AFHI11 FII - Cotação, Dividendos, Gráficos e Resultados - Investidor10</title>
    
    <meta name="description" content="Acompanhe o AFHI11. Cotação atualizada, histórico de dividendos (DY), resultados, gráficos, proventos e notícias sobre o fundo imobiliário Af Invest Cri - FII.">

    
    <link rel="canonical" href="https://investidor10.com.br/fiis/afhi11/">

    
    <meta name="viewport"
        content="width=device-width, initial-scale=1, minimum-scale=1, maximum-scale=1, user-scalable=no" />

    <link rel="icon" href="https://investidor10.com.br/assets/front/images/icon-32x32.ico" sizes="32x32" />

    <link rel="shortcut icon" href="https://investidor10.com.br/favicon.ico" ty

Progresso:   2%|▏         | 1/51 [00:02<02:11,  2.63s/it]

Código de Status: 200

Trecho do HTML (primeiros 1000 caracteres):
<!DOCTYPE html>
<html lang="pt-BR">

<head>
    <meta http-equiv="x-ua-compatible" content="ie=edge" />
    <meta charset="UTF-8" />
    <meta name="language" content="pt-BR" />
    <meta name="robots" content="index,follow">

    <title>ALZR11 FII - Cotação, Dividendos, Gráficos e Resultados - Investidor10</title>
    
    <meta name="description" content="Acompanhe o ALZR11. Cotação atualizada, histórico de dividendos (DY), resultados, gráficos, proventos e notícias sobre o fundo imobiliário Alianza Trust Renda ImobiliÁria - FII.">

    
    <link rel="canonical" href="https://investidor10.com.br/fiis/alzr11/">

    
    <meta name="viewport"
        content="width=device-width, initial-scale=1, minimum-scale=1, maximum-scale=1, user-scalable=no" />

    <link rel="icon" href="https://investidor10.com.br/assets/front/images/icon-32x32.ico" sizes="32x32" />

    <link rel="shortcut icon" href="https://investidor10.com.

...

Código de Status: 200

Trecho do HTML (primeiros 1000 caracteres):
<!DOCTYPE html>
<html lang="pt-BR">

<head>
    <meta http-equiv="x-ua-compatible" content="ie=edge" />
    <meta charset="UTF-8" />
    <meta name="language" content="pt-BR" />
    <meta name="robots" content="index,follow">

    <title>XPML11 FII - Cotação, Dividendos, Gráficos e Resultados - Investidor10</title>
    
    <meta name="description" content="Acompanhe o XPML11. Cotação atualizada, histórico de dividendos (DY), resultados, gráficos, proventos e notícias sobre o fundo imobiliário Xp Malls Fundos - FII.">

    
    <link rel="canonical" href="https://investidor10.com.br/fiis/xpml11/">

    
    <meta name="viewport"
        content="width=device-width, initial-scale=1, minimum-scale=1, maximum-scale=1, user-scalable=no" />

    <link rel="icon" href="https://investidor10.com.br/assets/front/images/icon-32x32.ico" sizes="32x32" />

    <link rel="shortcut icon" href="https://investidor10.com.br/favicon.ico" 

Progresso:  98%|█████████▊| 50/51 [02:19<00:04,  4.18s/it]

Código de Status: 200

Trecho do HTML (primeiros 1000 caracteres):
<!DOCTYPE html>
<html lang="pt-BR">

<head>
    <meta http-equiv="x-ua-compatible" content="ie=edge" />
    <meta charset="UTF-8" />
    <meta name="language" content="pt-BR" />
    <meta name="robots" content="index,follow">

    <title>ZAVI11 FII - Cotação, Dividendos, Gráficos e Resultados - Investidor10</title>
    
    <meta name="description" content="Acompanhe o ZAVI11. Cotação atualizada, histórico de dividendos (DY), resultados, gráficos, proventos e notícias sobre o fundo imobiliário Zavit Real Estate - FII.">

    
    <link rel="canonical" href="https://investidor10.com.br/fiis/zavi11/">

    
    <meta name="viewport"
        content="width=device-width, initial-scale=1, minimum-scale=1, maximum-scale=1, user-scalable=no" />

    <link rel="icon" href="https://investidor10.com.br/assets/front/images/icon-32x32.ico" sizes="32x32" />

    <link rel="shortcut icon" href="https://investidor10.com.br/favicon.ico

Progresso: 100%|██████████| 51/51 [02:21<00:00,  2.78s/it]



Indicadores salvos em: Dados_FIIs_Indicadores.xlsx
Rendimentos salvos em: Dados_FIIs_Rendimentos.xlsx
